# 01 — IEB Triad Math Simulation

Verifies the mathematics in `core/triad_math.py`:
- Sequence flattening \(x = \mathrm{vec}(X)\)
- Softmax & cross-entropy
- Coupling loss \(\mathcal{L}_c = -\log(w+\varepsilon)\)
- Total loss \(\mathcal{L} = \mathcal{L}_{CE} + \lambda \mathcal{L}_c\)

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from core.triad_math import (
    coupling_loss_weight,
    flatten_sequence_rows,
    numpy_cross_entropy,
    numpy_softmax,
    total_loss_scalar,
)
from core.feature_spec import FEATURE_DIM, SEQUENCE_LEN

print("ROOT", ROOT)
print("D=", FEATURE_DIM, "T=", SEQUENCE_LEN)

## 1. Flatten sequence

Left-pad short sequences to \(T\) frames, concatenate each \(D\)-dim row.

In [ ]:
short_seq = [[float(t)] * FEATURE_DIM for t in range(4)]
flat = flatten_sequence_rows(short_seq)
assert len(flat) == FEATURE_DIM * SEQUENCE_LEN
assert flat[0] == 0.0, "left padding"
print("flat dim", len(flat), "nonzero tail", sum(1 for v in flat if v))

## 2. Softmax & cross-entropy

\[
\pi_k = \frac{e^{z_k}}{\sum_j e^{z_j}}, \quad CE = -\log \pi_y
\]

In [ ]:
logits_np = [2.0, 0.5, -1.0]
pi_np = numpy_softmax(logits_np)
ce_np = numpy_cross_entropy(pi_np, target=0)

logits_pt = torch.tensor([logits_np])
from core.triad_torch import softmax_probs
pi_pt = softmax_probs(logits_pt)[0].numpy()
ce_pt = float(torch.nn.functional.cross_entropy(logits_pt, torch.tensor([0])))

print("numpy pi", np.round(pi_np, 4))
print("torch pi", np.round(pi_pt, 4))
print("CE numpy", ce_np, "torch", ce_pt)
assert np.allclose(pi_np, pi_pt, atol=1e-5)

## 3. Coupling loss from ethogram weight

Higher ethogram prior \(w\) → lower penalty.

In [ ]:
weights = np.linspace(0.05, 0.95, 10)
losses = [coupling_loss_weight(float(w)) for w in weights]
for w, L in zip(weights, losses):
    print(f"w={w:.2f}  L_c={L:.3f}")
assert coupling_loss_weight(0.9) < coupling_loss_weight(0.1)

## 4. Total loss composition

\[
\mathcal{L} = CE_I + CE_E + CE_B + \lambda \mathcal{L}_c
\]

In [ ]:
L = total_loss_scalar(
    ce_intent=0.4,
    ce_emotion=0.35,
    ce_behavior=0.25,
    couple_weight=0.85,
    confidence=0.92,
)
manual = 1.0 + 0.3 * coupling_loss_weight(0.85)
print("total_loss", L)
print("manual", manual)
assert abs(L - manual) < 1e-9
print("math verification OK")

## 5. Plot coupling curve (optional)

In [ ]:
import matplotlib.pyplot as plt

w_grid = np.linspace(0.01, 1.0, 100)
L_grid = [coupling_loss_weight(float(w)) for w in w_grid]
plt.figure(figsize=(6, 3))
plt.plot(w_grid, L_grid)
plt.xlabel("ethogram weight w")
plt.ylabel("L_c = -log(w+eps)")
plt.title("Coupling loss vs ethogram prior")
plt.grid(True, alpha=0.3)
plt.show()